# ex013_PHT3D_13

In [ ]:
import pandas as pd
from IPython.display import display

comparison_rows = []
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

CASE_DIR = Path.cwd()
INPUT_DIR = CASE_DIR / "input_data"
OUTPUT_DIR = CASE_DIR / "output"
plt.rcParams.update({"font.size": 9, "figure.dpi": 120})
FLOW_RATE = 0.00024
results = np.load(OUTPUT_DIR / "results.npy")
headings = (OUTPUT_DIR / "results_headings.txt").read_text(encoding="utf-8-sig").splitlines()
result_times = np.load(OUTPUT_DIR / "results_times.npy")
observations = np.loadtxt(INPUT_DIR / "observations.txt")
reference = np.load(INPUT_DIR / "PHT3D_13_results.npy")
volume_ml = result_times * FLOW_RATE * 1000000.0
reference_volume_ml = reference["time_days"] * FLOW_RATE * 1000000.0
outlet = results[:, :, -1]
panels = (
    ("Cl", "Cl", 4, "blue", (0, 0.025)),
    ("SO$_4^{2-}$", "S_6", 5, "blue", (0, 0.01)),
    ("Mg", "Mg", 1, "red", (0, 0.025)),
    ("HCO$_3^-$", "C_4", 3, "red", (0, 0.015)),
    ("pH", "pH", 6, "green", (4, 10)),
    ("Ca$^{2+}$", "Ca", 2, "green", (0, 0.006)),
)
fig, axes = plt.subplots(3, 2, figsize=(8.0, 7.0), sharex=True)
for axis, (title, heading, obs_col, color, ylim) in zip(axes.flat, panels, strict=False):
    model = outlet[:, headings.index(heading)]
    error = np.interp(reference["time_days"], result_times, model) - reference[heading]
    comparison_rows.append({"Variable": heading, "RMSE": np.sqrt(np.mean(error**2))})
    valid = observations[:, obs_col] > -9999
    axis.plot(
        observations[valid, 0],
        observations[valid, obs_col],
        ".",
        color=color,
        markersize=5,
        label="Appelo et al.",
    )
    axis.plot(volume_ml, model, "-", color=color, linewidth=1.5, label="MF6PQC")
    axis.plot(
        reference_volume_ml, reference[heading], "--", color="black", linewidth=1.0, label="PHT3D"
    )
    axis.set_title(title, pad=2)
    axis.set_xlim(-100, 800)
    axis.set_ylim(*ylim)
    axis.set_ylabel("mol/L" if heading != "pH" else "")
    axis.tick_params(direction="in", top=True, right=True)
axes[2, 0].set_xlabel("ml outflow")
axes[2, 1].set_xlabel("ml outflow")
axes[0, 0].legend(frameon=False, fontsize=8)
fig.tight_layout()
plt.show()
comparison = pd.DataFrame(comparison_rows)
comparison = comparison.set_index("Variable")
display(comparison.style.format({"RMSE": "{:.6g}"}).set_uuid("ex013_1"))